# Medical-record entity extraction: LoRA fine-tuned LLM vs prompting vs BioBERT
Real data: **MACCROBAT** (Caufield et al., 2019; CC BY 4.0) — 200 PubMed Central clinical case reports
with manual expert annotations. Task: pull the facts a personal-injury medical chronology needs
(diagnoses/injuries, symptoms, body regions, diagnostic and therapeutic procedures, medications, dates, durations).

**Pre-registered setup (fixed before seeing test results):** document-level 70/10/20 split (seed 13);
3-sentence chunks; strict + relaxed set-based F1 per chunk; paired document-level bootstrap (2,000 resamples).
Run on **Kaggle, GPU T4 x2 (one GPU used), Internet ON**. Runtime ≈ 2–2.5 h.

In [ ]:
!pip -q uninstall -y torchao  # Kaggle ships torchao 0.10; current peft rejects <0.16 and we do not use it
!pip -q install -U peft sklearn-crfsuite
import torch, transformers, peft, sys, platform
print(torch.__version__, transformers.__version__, peft.__version__, platform.python_version())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — enable T4 in Settings")

In [ ]:
%%writefile mre.py
"""Core utilities for medical-record entity extraction on MACCROBAT.

Data loading (brat standoff), document-level splits, chunking, scoring
(strict + relaxed set-based F1, hallucination rate), lexicon and CRF
baselines, LLM prompt/JSON helpers, and paired document-level bootstrap.
Everything here is CPU-only and dependency-light so it can be unit tested.
"""
from __future__ import annotations

import json
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path

# Entity types kept: the ones a personal-injury medical chronology needs.
TYPES = [
    "Disease_disorder",       # diagnoses / injuries
    "Sign_symptom",           # complaints, findings
    "Biological_structure",   # body region
    "Diagnostic_procedure",   # imaging, exams, tests
    "Therapeutic_procedure",  # surgery, therapy, treatment
    "Medication",
    "Date",
    "Duration",
]
TYPE_KEYS = {t: t.lower() for t in TYPES}  # JSON keys used in LLM output
KEY_TYPES = {v: k for k, v in TYPE_KEYS.items()}


@dataclass
class Span:
    type: str
    start: int
    end: int
    text: str
    discontinuous: bool = False


@dataclass
class Doc:
    doc_id: str
    text: str
    spans: list[Span] = field(default_factory=list)


@dataclass
class Chunk:
    doc_id: str
    idx: int
    start: int
    end: int
    text: str
    spans: list[Span]


# ----------------------------------------------------------------- loading
def parse_ann(ann_text: str, doc_text: str, keep=TYPES) -> list[Span]:
    spans = []
    for line in ann_text.splitlines():
        if not line.startswith("T"):
            continue  # skip events (E), relations (R), attributes (A), notes (#)
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        head, surface = parts[1], parts[2]
        typ, offs = head.split(" ", 1)
        if typ not in keep:
            continue
        frags = [tuple(map(int, f.split())) for f in offs.split(";")]
        start, end = min(f[0] for f in frags), max(f[1] for f in frags)
        disc = len(frags) > 1
        if not disc and doc_text[start:end] != surface:
            # offsets disagree with text: trust the surface string if locatable
            loc = doc_text.find(surface, max(0, start - 20))
            if loc == -1:
                continue
            start, end = loc, loc + len(surface)
        spans.append(Span(typ, start, end, surface, disc))
    spans.sort(key=lambda s: (s.start, -s.end))
    return spans


def load_brat_dir(root: str | Path) -> list[Doc]:
    root = Path(root)
    docs = []
    for txt in sorted(root.rglob("*.txt")):
        ann = txt.with_suffix(".ann")
        if not ann.exists():
            continue
        text = txt.read_text(encoding="utf-8")
        docs.append(Doc(txt.stem, text, parse_ann(ann.read_text(encoding="utf-8"), text)))
    # MACCROBAT2018 and 2020 releases can share PubMed IDs: keep one copy
    seen, uniq = set(), []
    for d in docs:
        if d.doc_id not in seen:
            seen.add(d.doc_id)
            uniq.append(d)
    return uniq


def split_docs(docs, seed=13, frac=(0.7, 0.1, 0.2)):
    ids = sorted(d.doc_id for d in docs)
    rng = random.Random(seed)
    rng.shuffle(ids)
    n = len(ids)
    a, b = int(frac[0] * n), int((frac[0] + frac[1]) * n)
    part = {i: "train" for i in ids[:a]} | {i: "val" for i in ids[a:b]} | {i: "test" for i in ids[b:]}
    out = {"train": [], "val": [], "test": []}
    for d in docs:
        out[part[d.doc_id]].append(d)
    return out


def chunk_doc(doc: Doc, max_lines=3, max_chars=900) -> list[Chunk]:
    """Group consecutive sentence lines; spans crossing a boundary are dropped
    from both chunks (counted by `cross_boundary_spans`)."""
    lines, pos = [], 0
    for ln in doc.text.split("\n"):
        lines.append((pos, pos + len(ln)))
        pos += len(ln) + 1
    groups, cur = [], []
    for s, e in lines:
        if e == s:
            continue
        if cur and (len(cur) >= max_lines or e - cur[0][0] > max_chars):
            groups.append(cur)
            cur = []
        cur.append((s, e))
    if cur:
        groups.append(cur)
    chunks = []
    for i, g in enumerate(groups):
        s, e = g[0][0], g[-1][1]
        inside = [Span(sp.type, sp.start - s, sp.end - s, sp.text, sp.discontinuous)
                  for sp in doc.spans if sp.start >= s and sp.end <= e]
        chunks.append(Chunk(doc.doc_id, i, s, e, doc.text[s:e], inside))
    return chunks


def cross_boundary_spans(doc: Doc, chunks: list[Chunk]) -> int:
    return len(doc.spans) - sum(len(c.spans) for c in chunks)


# ----------------------------------------------------------------- scoring
_ws = re.compile(r"\s+")


def norm(s: str) -> str:
    s = _ws.sub(" ", s.lower()).strip()
    return s.strip(" .,;:()[]\"'")


def gold_set(chunk: Chunk) -> set[tuple[str, str]]:
    return {(sp.type, norm(sp.text)) for sp in chunk.spans if norm(sp.text)}


def _toks(s):
    return set(re.findall(r"\w+", s))


def relaxed_match(a: str, b: str) -> bool:
    if a == b or a in b or b in a:
        return True
    ta, tb = _toks(a), _toks(b)
    return bool(ta and tb) and len(ta & tb) / len(ta | tb) >= 0.5


def match_counts(pred: set, gold: set, relaxed=False):
    """Return tp, fp, fn per type. Relaxed uses greedy one-to-one matching."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for t in TYPES:
        p = sorted(x for (ty, x) in pred if ty == t)
        g = sorted(x for (ty, x) in gold if ty == t)
        if not relaxed:
            k = len(set(p) & set(g))
            tp[t] += k
            fp[t] += len(p) - k
            fn[t] += len(g) - k
            continue
        used = set()
        for x in p:
            hit = next((j for j, y in enumerate(g) if j not in used and relaxed_match(x, y)), None)
            if hit is None:
                fp[t] += 1
            else:
                used.add(hit)
                tp[t] += 1
        fn[t] += len(g) - len(used)
    return tp, fp, fn


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def score(chunks: list[Chunk], preds: dict, relaxed=False):
    """preds: {(doc_id, idx): set[(type, normtext)]}"""
    TP, FP, FN = Counter(), Counter(), Counter()
    for c in chunks:
        tp, fp, fn = match_counts(preds.get((c.doc_id, c.idx), set()), gold_set(c), relaxed)
        TP += tp; FP += fp; FN += fn
    per = {t: dict(zip(("precision", "recall", "f1"), prf(TP[t], FP[t], FN[t])),
                   support=TP[t] + FN[t]) for t in TYPES}
    p, r, f = prf(sum(TP.values()), sum(FP.values()), sum(FN.values()))
    macro = sum(per[t]["f1"] for t in TYPES) / len(TYPES)
    return {"micro_precision": p, "micro_recall": r, "micro_f1": f, "macro_f1": macro, "per_type": per}


def doc_counts(chunks, preds, relaxed=False):
    """Per-document summed tp/fp/fn (all types) for bootstrap."""
    out = defaultdict(lambda: [0, 0, 0])
    for c in chunks:
        tp, fp, fn = match_counts(preds.get((c.doc_id, c.idx), set()), gold_set(c), relaxed)
        v = out[c.doc_id]
        v[0] += sum(tp.values()); v[1] += sum(fp.values()); v[2] += sum(fn.values())
    return dict(out)


def paired_bootstrap(chunks, preds_a, preds_b, n=2000, seed=0, relaxed=False):
    """Doc-level paired bootstrap of micro-F1(a) - micro-F1(b)."""
    da, db = doc_counts(chunks, preds_a, relaxed), doc_counts(chunks, preds_b, relaxed)
    ids = sorted(set(da) | set(db))
    rng = random.Random(seed)

    def f1(d, sample):
        tp = sum(d.get(i, [0, 0, 0])[0] for i in sample)
        fp = sum(d.get(i, [0, 0, 0])[1] for i in sample)
        fn = sum(d.get(i, [0, 0, 0])[2] for i in sample)
        return prf(tp, fp, fn)[2]

    obs = f1(da, ids) - f1(db, ids)
    diffs = sorted(f1(da, s) - f1(db, s) for s in
                   ([rng.choice(ids) for _ in ids] for _ in range(n)))
    lo, hi = diffs[int(0.025 * n)], diffs[int(0.975 * n) - 1]
    return {"diff": obs, "ci95": [lo, hi], "n_docs": len(ids), "n_boot": n}


def hallucination_rate(chunks, raw_preds):
    """Share of predicted strings not found (normalized) in the source chunk."""
    by = {(c.doc_id, c.idx): norm(c.text) for c in chunks}
    tot = bad = 0
    for k, items in raw_preds.items():
        src = by.get(k, "")
        for _, s in items:
            tot += 1
            bad += norm(s) not in src
    return bad / tot if tot else 0.0


# ------------------------------------------------------- lexicon baseline
def build_lexicon(chunks, min_len=3):
    votes = defaultdict(Counter)
    for c in chunks:
        for sp in c.spans:
            n = norm(sp.text)
            if len(n) >= min_len:
                votes[n][sp.type] += 1
    return {k: v.most_common(1)[0][0] for k, v in votes.items()}


def lexicon_predict(chunk, lex):
    text = chunk.text.lower()
    taken = [False] * len(text)
    out = set()
    for term in sorted(lex, key=len, reverse=True):
        for m in re.finditer(r"(?<!\w)" + re.escape(term) + r"(?!\w)", text):
            if not any(taken[m.start():m.end()]):
                for i in range(m.start(), m.end()):
                    taken[i] = True
                out.add((lex[term], term))
    return out


# ----------------------------------------------------------- CRF baseline
_tok = re.compile(r"\w+|[^\w\s]")


def tokenize(text):
    return [(m.group(), m.start(), m.end()) for m in _tok.finditer(text)]


def bio_labels(chunk):
    """BIO tags from char spans; longer spans win overlaps. Returns tags and
    the number of gold spans that could not be encoded (nested/overlapping)."""
    toks = tokenize(chunk.text)
    tags = ["O"] * len(toks)
    dropped = 0
    for sp in sorted(chunk.spans, key=lambda s: -(s.end - s.start)):
        idx = [i for i, (_, s, e) in enumerate(toks) if s >= sp.start and e <= sp.end]
        if not idx or any(tags[i] != "O" for i in idx):
            dropped += 1
            continue
        tags[idx[0]] = "B-" + sp.type
        for i in idx[1:]:
            tags[i] = "I-" + sp.type
    return toks, tags, dropped


def tags_to_set(text, toks, tags):
    out, cur = set(), None
    for (w, s, e), t in zip(toks, tags):
        if t.startswith("B-") or (t.startswith("I-") and (cur is None or cur[0] != t[2:])):
            if cur:
                out.add((cur[0], norm(text[cur[1]:cur[2]])))
            cur = [t[2:], s, e]
        elif t.startswith("I-"):
            cur[2] = e
        else:
            if cur:
                out.add((cur[0], norm(text[cur[1]:cur[2]])))
            cur = None
    if cur:
        out.add((cur[0], norm(text[cur[1]:cur[2]])))
    return {x for x in out if x[1]}


def _wfeat(w):
    return {"w": w.lower(), "suf3": w[-3:].lower(), "pre3": w[:3].lower(),
            "title": w.istitle(), "upper": w.isupper(), "digit": w.isdigit(),
            "hasdigit": any(ch.isdigit() for ch in w), "len": min(len(w), 12)}


def crf_features(toks):
    feats = []
    for i, (w, _, _) in enumerate(toks):
        f = {"bias": 1.0, **_wfeat(w)}
        for d in (-2, -1, 1, 2):
            j = i + d
            if 0 <= j < len(toks):
                for k, v in _wfeat(toks[j][0]).items():
                    if k in ("w", "suf3", "title", "digit"):
                        f[f"{d}:{k}"] = v
            else:
                f[f"{d}:pad"] = True
        feats.append(f)
    return feats


# ---------------------------------------------------------- LLM helpers
SYSTEM = ("You extract medical facts from clinical text for personal-injury case review. "
          "Copy each mention exactly as written in the text. Do not paraphrase, infer, or add "
          "anything not in the text. Return only JSON.")


def instruction(text):
    keys = ", ".join(TYPE_KEYS[t] for t in TYPES)
    return (f"Extract every mention of these types: {keys}.\n"
            "Return a JSON object with exactly those keys; each value is a list of exact "
            "substrings from the text (empty list if none).\n\nText:\n" + text)


def target_json(chunk):
    out = {TYPE_KEYS[t]: [] for t in TYPES}
    for sp in sorted(chunk.spans, key=lambda s: s.start):
        lst = out[TYPE_KEYS[sp.type]]
        if sp.text not in lst:
            lst.append(sp.text)
    return json.dumps(out, ensure_ascii=False)


def parse_llm_json(s):
    """Return (set of (type, normtext), list of (type, raw), valid_json)."""
    m = re.search(r"\{.*\}", s, re.S)
    if not m:
        return set(), [], False
    try:
        obj = json.loads(m.group())
    except json.JSONDecodeError:
        return set(), [], False
    if not isinstance(obj, dict):
        return set(), [], False
    raw = []
    for k, v in obj.items():
        t = KEY_TYPES.get(str(k).lower())
        if t is None or not isinstance(v, list):
            continue
        raw += [(t, x) for x in v if isinstance(x, str) and x.strip()]
    return {(t, norm(x)) for t, x in raw if norm(x)}, raw, True

In [ ]:
# ---- Download MACCROBAT from figshare (article 9764942) and load ----
import json, os, zipfile, io, urllib.request, glob, pathlib, collections
import mre, importlib; importlib.reload(mre)
os.makedirs("data", exist_ok=True); os.makedirs("results", exist_ok=True)
UA = {"User-Agent": "Mozilla/5.0 (research notebook)"}
get = lambda u: urllib.request.urlopen(urllib.request.Request(u, headers=UA)).read()
api = json.loads(get("https://api.figshare.com/v2/articles/9764942"))
print("figshare version:", api.get("version"), [f["name"] for f in api["files"]])
def extract(blob, dest):
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for n in z.namelist():
            if n.endswith(".zip"):
                extract(z.read(n), os.path.join(dest, pathlib.Path(n).stem))
            elif not n.endswith("/"):
                p = os.path.join(dest, n); os.makedirs(os.path.dirname(p), exist_ok=True)
                open(p, "wb").write(z.read(n))
for f in api["files"]:
    blob = get(f["download_url"])
    if f["name"].endswith(".zip"):
        extract(blob, os.path.join("data", pathlib.Path(f["name"]).stem))
# Prefer the 2020 release (improved annotation consistency) if both are present
ann_dirs = sorted({os.path.dirname(p) for p in glob.glob("data/**/*.ann", recursive=True)})
print("annotation dirs:", ann_dirs)
pick = [d for d in ann_dirs if "2020" in d] or ann_dirs
docs = []
for d in pick: docs += mre.load_brat_dir(d)
seen=set(); docs=[x for x in docs if not (x.doc_id in seen or seen.add(x.doc_id))]
cnt = collections.Counter(s.type for d in docs for s in d.spans)
stats = {"source_dirs": pick, "n_docs": len(docs), "n_spans": sum(cnt.values()),
         "spans_per_type": dict(cnt),
         "discontinuous_spans": sum(s.discontinuous for d in docs for s in d.spans)}
print(json.dumps(stats, indent=1))

In [ ]:
# ---- Split and chunk ----
sp = mre.split_docs(docs, seed=13)
CH = {k: [c for d in v for c in mre.chunk_doc(d)] for k, v in sp.items()}
cross = {k: sum(mre.cross_boundary_spans(d, mre.chunk_doc(d)) for d in v) for k, v in sp.items()}
stats.update({"split_docs": {k: len(v) for k, v in sp.items()},
              "split_chunks": {k: len(v) for k, v in CH.items()},
              "spans_lost_at_chunk_boundaries": cross,
              "test_doc_ids": sorted(d.doc_id for d in sp["test"])})
print({k: stats[k] for k in ("split_docs", "split_chunks", "spans_lost_at_chunk_boundaries")})
test = CH["test"]; RESULTS = {}; PRED = {}
def record(name, preds, extra=None):
    PRED[name] = preds
    RESULTS[name] = {"strict": mre.score(test, preds), "relaxed": mre.score(test, preds, relaxed=True), **(extra or {})}
    r = RESULTS[name]
    print(f"{name:28s} strict micro-F1 {r['strict']['micro_f1']:.3f} | relaxed {r['relaxed']['micro_f1']:.3f}")

In [ ]:
# ---- Baseline 1: train-set lexicon lookup ----
lex = mre.build_lexicon(CH["train"])
record("lexicon", {(c.doc_id, c.idx): mre.lexicon_predict(c, lex) for c in test}, {"lexicon_size": len(lex)})

In [ ]:
# ---- Baseline 2: CRF (sklearn-crfsuite) BIO tagger ----
import sklearn_crfsuite
def xy(chs):
    X, Y, drop = [], [], 0
    for c in chs:
        toks, tags, d = mre.bio_labels(c); drop += d
        X.append(mre.crf_features(toks)); Y.append(tags)
    return X, Y, drop
Xtr, Ytr, drop_tr = xy(CH["train"])
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=200, all_possible_transitions=True)
crf.fit(Xtr, Ytr)
pc = {}
for c in test:
    toks = mre.tokenize(c.text)
    pc[(c.doc_id, c.idx)] = mre.tags_to_set(c.text, toks, crf.predict_single(mre.crf_features(toks)))
record("crf", pc, {"train_spans_not_bio_encodable": drop_tr})

In [ ]:
# ---- Baseline 3: BioBERT token classifier (fine-tuned, best epoch on val) ----
import numpy as np, random, time, torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
torch.manual_seed(0); random.seed(0); np.random.seed(0)
BERT = "dmis-lab/biobert-base-cased-v1.2"
LABELS = ["O"] + [f"{p}-{t}" for t in mre.TYPES for p in "BI"]; L2I = {l: i for i, l in enumerate(LABELS)}
btok = AutoTokenizer.from_pretrained(BERT)
def enc(c):
    e = btok(c.text, truncation=True, max_length=512, return_offsets_mapping=True)
    offs = e["offset_mapping"]; lab = [-100 if s == t == 0 else 0 for s, t in offs]
    for sp_ in sorted(c.spans, key=lambda s: -(s.end - s.start)):
        idx = [i for i, (s, t) in enumerate(offs) if not s == t == 0 and s >= sp_.start and t <= sp_.end]
        if not idx or any(lab[i] != 0 for i in idx): continue
        lab[idx[0]] = L2I["B-" + sp_.type]
        for i in idx[1:]: lab[i] = L2I["I-" + sp_.type]
    return e["input_ids"], lab, offs
def bert_predict(model, chs, bs=32):
    model.eval(); out = {}
    for i in range(0, len(chs), bs):
        b = chs[i:i+bs]; e = btok([c.text for c in b], truncation=True, max_length=512, padding=True,
                                  return_offsets_mapping=True, return_tensors="pt")
        offs = e.pop("offset_mapping")
        with torch.no_grad(): lg = model(**{k: v.cuda() for k, v in e.items()}).logits.argmax(-1).cpu()
        for c, o, p, m in zip(b, offs, lg, e["attention_mask"]):
            toks, tags = [], []
            for (s, t), y, mm in zip(o.tolist(), p.tolist(), m.tolist()):
                if mm and not s == t == 0: toks.append((c.text[s:t], s, t)); tags.append(LABELS[y])
            out[(c.doc_id, c.idx)] = mre.tags_to_set(c.text, toks, tags)
    return out
tr = [enc(c) for c in CH["train"]]
bert = AutoModelForTokenClassification.from_pretrained(BERT, num_labels=len(LABELS)).cuda()
opt = torch.optim.AdamW(bert.parameters(), lr=3e-5, weight_decay=0.01)
best, best_state, curve = -1, None, []
t0 = time.time()
for ep in range(10):
    bert.train(); random.shuffle(tr)
    for i in range(0, len(tr), 8):
        b = tr[i:i+8]; n = max(len(x[0]) for x in b)
        ids = torch.tensor([x[0] + [btok.pad_token_id] * (n - len(x[0])) for x in b]).cuda()
        lab = torch.tensor([x[1] + [-100] * (n - len(x[1])) for x in b]).cuda()
        loss = bert(input_ids=ids, attention_mask=(ids != btok.pad_token_id).long(), labels=lab).loss
        loss.backward(); opt.step(); opt.zero_grad()
    vf = mre.score(CH["val"], bert_predict(bert, CH["val"]))["micro_f1"]; curve.append(vf)
    print(f"epoch {ep+1} val strict micro-F1 {vf:.3f}")
    if vf > best: best, best_ep, best_state = vf, ep + 1, {k: v.detach().cpu().clone() for k, v in bert.state_dict().items()}
bert.load_state_dict(best_state)
t1 = time.time(); pb = bert_predict(bert, test); lat = (time.time() - t1) / len(test)
record("biobert", pb, {"best_epoch": best_ep, "val_curve": curve, "train_minutes": (t1 - t0) / 60,
                       "sec_per_chunk": lat})
# keep `bert` loaded: reused in the OCR robustness cell

In [ ]:
# ---- LLM: Qwen2.5-1.5B-Instruct, zero-shot and 2-shot prompting ----
from transformers import AutoModelForCausalLM
LLM = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM); tok.padding_side = "left"
llm = AutoModelForCausalLM.from_pretrained(LLM, torch_dtype=torch.float16).cuda().eval()
# 2 fixed few-shot examples: the two shortest train chunks that contain >= 4 spans
shots = sorted([c for c in CH["train"] if len(c.spans) >= 4], key=lambda c: len(c.text))[:2]
def messages(c, fewshot=False):
    m = [{"role": "system", "content": mre.SYSTEM}]
    if fewshot:
        for s in shots: m += [{"role": "user", "content": mre.instruction(s.text)},
                              {"role": "assistant", "content": mre.target_json(s)}]
    return m + [{"role": "user", "content": mre.instruction(c.text)}]
def generate(model, chs, fewshot=False, bs=16, max_new=512):
    raw_out, sets, raws, valid, trunc = {}, {}, {}, 0, 0
    t0 = time.time()
    for i in range(0, len(chs), bs):
        b = chs[i:i+bs]
        prompts = [tok.apply_chat_template(messages(c, fewshot), tokenize=False, add_generation_prompt=True) for c in b]
        e = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            g = model.generate(**e, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
        for c, seq in zip(b, g[:, e["input_ids"].shape[1]:]):
            txt = tok.decode(seq, skip_special_tokens=True); k = (c.doc_id, c.idx)
            trunc += int((seq != tok.eos_token_id).sum().item() >= max_new)
            s, r, ok = mre.parse_llm_json(txt); valid += ok
            raw_out[k], sets[k], raws[k] = txt, s, r
    n = len(chs)
    return sets, raws, raw_out, {"json_valid_rate": valid / n, "truncated_rate": trunc / n,
                                 "sec_per_chunk": (time.time() - t0) / n}
for name, fs in [("qwen1.5b_zeroshot", False), ("qwen1.5b_2shot", True)]:
    s, r, raw, ex = generate(llm, test, fewshot=fs)
    ex["hallucination_rate"] = mre.hallucination_rate(test, r)
    record(name, s, ex)
    json.dump({f"{k[0]}#{k[1]}": v for k, v in raw.items()}, open(f"results/raw_{name}.json", "w"))

In [ ]:
# ---- LoRA fine-tune (PEFT) on train chunks; loss on assistant tokens only ----
from peft import LoraConfig, get_peft_model
torch.manual_seed(0); random.seed(0)
lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(llm, lcfg)
for n_, p in model.named_parameters():
    if p.requires_grad: p.data = p.data.float()  # fp32 adapters, fp16 frozen base
model.gradient_checkpointing_enable(); model.enable_input_require_grads(); model.config.use_cache = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
def sft_example(c):
    prompt = tok.apply_chat_template(messages(c), tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    t_ids = tok(mre.target_json(c) + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = (p_ids + t_ids)[:1024]
    return ids, ([-100] * len(p_ids) + t_ids)[:1024]
train_ex = [sft_example(c) for c in CH["train"]]
EPOCHS, ACC, LR = 3, 8, 2e-4
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
total = EPOCHS * len(train_ex) // ACC
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 20) * max(0.0, 1 - s / total))
scaler = torch.amp.GradScaler("cuda"); losses = []; t0 = time.time(); step = 0
model.train()
for ep in range(EPOCHS):
    random.shuffle(train_ex); run = 0.0
    for i, (ids, lab) in enumerate(train_ex):
        x = torch.tensor([ids]).cuda(); y = torch.tensor([lab]).cuda()
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(input_ids=x, labels=y).loss / ACC
        scaler.scale(loss).backward(); run += loss.item()
        if (i + 1) % ACC == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step(); step += 1
    losses.append(run * ACC / len(train_ex)); print(f"epoch {ep+1} mean train loss {losses[-1]:.4f}")
train_min = (time.time() - t0) / 60
model.gradient_checkpointing_disable(); model.eval(); model.config.use_cache = True
vs, vr, _, vex = generate(model, CH["val"])
val_f1 = mre.score(CH["val"], vs)["micro_f1"]
s, r, raw, ex = generate(model, test)
ex.update({"hallucination_rate": mre.hallucination_rate(test, r), "train_minutes": train_min,
           "epoch_train_loss": losses, "trainable_params": trainable, "val_strict_micro_f1": val_f1,
           "hparams": {"r": 16, "alpha": 32, "dropout": 0.05, "epochs": EPOCHS, "lr": LR, "grad_accum": ACC, "max_len": 1024}})
record("qwen1.5b_lora", s, ex)
json.dump({f"{k[0]}#{k[1]}": v for k, v in raw.items()}, open("results/raw_qwen1.5b_lora.json", "w"))
model.save_pretrained("results/lora_adapter")

## OCR robustness (simulated scans)
Real medical records often arrive as scans. Each test chunk is rendered as a page image at three pre-set
degradation levels (blur, speckle noise, skew, JPEG, downscale), OCR'd with Tesseract, and re-scored against
the **original** gold annotations. Degradation is synthetic, applied to real text; these are not real scans.

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1; pip -q install pytesseract; tesseract --version | head -1

In [ ]:
%%writefile ocr.py
"""Simulated-scan OCR robustness: render real case-report text as a page
image, apply scanner-style degradation, OCR it with Tesseract, measure CER.

The degradation is synthetic (applied to real text); these are not real
scanned records. Levels were calibrated on a hand-written sample (target
roughly 3-5% and ~10% CER) before any MACCROBAT result existed, then frozen.
"""
from __future__ import annotations

import io
import random
import textwrap

from PIL import Image, ImageDraw, ImageFilter, ImageFont

LEVELS = {
    "clean":  dict(blur=0.0, noise=0.00, rotate=0.0, jpeg=95, scale=1.0),
    "mild":   dict(blur=0.4, noise=0.010, rotate=0.5, jpeg=70, scale=0.9),
    "heavy":  dict(blur=0.6, noise=0.018, rotate=1.0, jpeg=45, scale=0.8),
}
_FONTS = ["/usr/share/fonts/truetype/dejavu/DejaVuSerif.ttf",
          "/usr/share/fonts/truetype/liberation/LiberationSerif-Regular.ttf"]


def _font(size=22):
    for p in _FONTS:
        try:
            return ImageFont.truetype(p, size)
        except OSError:
            continue
    return ImageFont.load_default(size=size)


def render(text: str, level: str, seed: int = 0, width_chars: int = 80) -> Image.Image:
    cfg, rng = LEVELS[level], random.Random(seed)
    font = _font()
    lines = []
    for para in text.split("\n"):
        lines += textwrap.wrap(para, width_chars) or [""]
    lh = 30
    img = Image.new("L", (1000, 60 + lh * len(lines)), 255)
    d = ImageDraw.Draw(img)
    for i, ln in enumerate(lines):
        d.text((40, 30 + i * lh), ln, fill=0, font=font)
    if cfg["rotate"]:
        img = img.rotate(rng.uniform(-cfg["rotate"], cfg["rotate"]), expand=True, fillcolor=255)
    if cfg["scale"] != 1.0:
        w, h = img.size
        img = img.resize((int(w * cfg["scale"]), int(h * cfg["scale"])), Image.BILINEAR)
    if cfg["blur"]:
        img = img.filter(ImageFilter.GaussianBlur(cfg["blur"]))
    if cfg["noise"]:
        px = img.load()
        w, h = img.size
        for _ in range(int(cfg["noise"] * w * h)):
            x, y = rng.randrange(w), rng.randrange(h)
            px[x, y] = 0 if rng.random() < 0.5 else 255
    buf = io.BytesIO()
    img.save(buf, "JPEG", quality=cfg["jpeg"])
    return Image.open(io.BytesIO(buf.getvalue()))


def ocr(img: Image.Image) -> str:
    import pytesseract
    return pytesseract.image_to_string(img, config="--psm 6").strip()


def _lev(a: str, b: str) -> int:
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


def cer(ref: str, hyp: str) -> float:
    """Character error rate on whitespace-collapsed text."""
    r, h = " ".join(ref.split()), " ".join(hyp.split())
    return _lev(r, h) / max(1, len(r))


def ocr_job(args):
    """(key, text, level, seed) -> (key, level, ocr_text, cer). Pool-friendly."""
    key, text, level, seed = args
    out = ocr(render(text, level, seed))
    return key, level, out, cer(text, out)

In [ ]:
import ocr, importlib; importlib.reload(ocr)
from multiprocessing.pool import ThreadPool  # threads, not fork: forking a CUDA/torch process can deadlock; tesseract runs as a subprocess anyway
os.environ["OMP_THREAD_LIMIT"] = "1"  # one OpenMP thread per tesseract call
os.makedirs("results/ocr_samples", exist_ok=True)
jobs = [((c.doc_id, c.idx), c.text, lv, i) for i, c in enumerate(test) for lv in ocr.LEVELS]
t0 = time.time()
with ThreadPool(4) as pool: outs = pool.map(ocr.ocr_job, jobs, chunksize=8)
print(f"OCR {len(jobs)} pages in {(time.time()-t0)/60:.1f} min")
for lv in ocr.LEVELS: ocr.render(test[0].text, lv, 0).save(f"results/ocr_samples/{test[0].doc_id}_{lv}.jpg")
OCR = {lv: {} for lv in ocr.LEVELS}; CER = {lv: [] for lv in ocr.LEVELS}
for k, lv, txt, c in outs: OCR[lv][k] = txt; CER[lv].append(c)
OCR_RESULTS = {}; OCR_PRED = {}
for lv in ocr.LEVELS:
    och = [mre.Chunk(c.doc_id, c.idx, c.start, c.end, OCR[lv][(c.doc_id, c.idx)], c.spans) for c in test]
    preds = {"lexicon": {(c.doc_id, c.idx): mre.lexicon_predict(c, lex) for c in och}}
    pc = {}
    for c in och:
        toks = mre.tokenize(c.text); pc[(c.doc_id, c.idx)] = mre.tags_to_set(c.text, toks, crf.predict_single(mre.crf_features(toks)))
    preds["crf"] = pc
    preds["biobert"] = bert_predict(bert, och)
    s_, r_, _, ex_ = generate(model, och); preds["qwen1.5b_lora"] = s_
    OCR_PRED[lv] = preds
    OCR_RESULTS[lv] = {"mean_cer": float(np.mean(CER[lv])), "median_cer": float(np.median(CER[lv])),
                       "lora_json_valid_rate": ex_["json_valid_rate"],
                       "systems": {n_: {m: mre.score(test, p_, relaxed=(m == "relaxed"))["micro_f1"]
                                        for m in ("strict", "relaxed")} for n_, p_ in preds.items()}}
    print(lv, f"CER {OCR_RESULTS[lv]['mean_cer']:.3f}",
          {n_: round(v["strict"], 3) for n_, v in OCR_RESULTS[lv]["systems"].items()})
OCR_BOOT = {f"{n_}_{lv}_minus_original": mre.paired_bootstrap(test, OCR_PRED[lv][n_], PRED[n_])
            for lv in ("mild", "heavy") for n_ in ("biobert", "qwen1.5b_lora")}
print(json.dumps({k: [round(v["diff"], 4), [round(x, 4) for x in v["ci95"]]] for k, v in OCR_BOOT.items()}, indent=1))

In [ ]:
# ---- Paired bootstrap, tables, save everything ----
import pandas as pd
pairs = [("qwen1.5b_lora", "qwen1.5b_2shot"), ("qwen1.5b_lora", "qwen1.5b_zeroshot"),
         ("qwen1.5b_lora", "biobert"), ("biobert", "crf"), ("crf", "lexicon")]
boot = {f"{a}_minus_{b}": {m: mre.paired_bootstrap(test, PRED[a], PRED[b], relaxed=(m == "relaxed"))
                           for m in ("strict", "relaxed")} for a, b in pairs}
rows = []
for n_, r_ in RESULTS.items():
    rows.append({"system": n_, **{f"{m}_{k}": round(r_[m][k], 4) for m in ("strict", "relaxed")
                                  for k in ("micro_precision", "micro_recall", "micro_f1", "macro_f1")},
                 **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r_.items()
                    if k in ("json_valid_rate", "truncated_rate", "hallucination_rate", "sec_per_chunk")}})
summary = pd.DataFrame(rows); summary.to_csv("results/summary.csv", index=False)
per = pd.DataFrame([{"system": n_, "type": t, **{f"strict_{k}": round(v, 4) if isinstance(v, float) else v
                     for k, v in r_["strict"]["per_type"][t].items()},
                     "relaxed_f1": round(r_["relaxed"]["per_type"][t]["f1"], 4)}
                    for n_, r_ in RESULTS.items() for t in mre.TYPES])
per.to_csv("results/per_type.csv", index=False)
env = {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "gpu": torch.cuda.get_device_name(0)}
json.dump({"data": stats, "results": RESULTS, "bootstrap": boot, "ocr": OCR_RESULTS,
           "ocr_bootstrap": OCR_BOOT, "env": env},
          open("results/metrics.json", "w"), indent=1, default=float)
display(summary)
print(json.dumps({k: {m: {"diff": round(v[m]["diff"], 4), "ci95": [round(x, 4) for x in v[m]["ci95"]]}
                      for m in v} for k, v in boot.items()}, indent=1))
!cd results && zip -qr ../results.zip . && ls -la ..

**Send back:** `results/metrics.json`, `results/summary.csv`, `results/per_type.csv`
(or the whole `results.zip` from the Output panel). Nothing is claimed until these numbers come back.